# TP 1 — Premiers pas avec le NLP

**Module NLP — Formation Machine Learning**

Ce TP est une **première prise en main** du NLP (Natural Language Processing). L'objectif n'est pas encore de construire une application complète (ça viendra dans un futur projet), mais de **manipuler soi-même**, sur des exemples simples, les notions vues en cours :

`Texte brut → Prétraitement → Vectorisation (Bag of Words / TF-IDF)`

## Objectifs pédagogiques

- Comprendre pourquoi le texte est difficile à traiter pour une machine.
- Nettoyer et prétraiter du texte brut (minuscules, tokenisation, stopwords, lemmatisation).
- Transformer du texte en vecteurs de nombres avec le **Bag of Words**.
- Aller plus loin avec le **TF-IDF** et comprendre en quoi il est plus intelligent que le Bag of Words.
- Découvrir concrètement les principales librairies Python du NLP : **scikit-learn**, **NLTK**, **spaCy**, **Hugging Face**.
- S'entraîner sur ses propres phrases via les **exercices** en fin de notebook.

> 💡 Ce notebook est conçu pour être exécuté directement dans **Google Colab**, cellule par cellule (`Shift + Entrée`). Aucun fichier externe n'est nécessaire : on travaille sur des phrases simples écrites directement dans le notebook.


## Étape 0 — Installation et imports


In [ ]:
# Ces librairies sont déjà installées sur Google Colab par défaut.
# Si besoin : !pip install scikit-learn pandas

import pandas as pd
import re

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

pd.set_option("display.max_colwidth", None)
print("Librairies importées avec succès ✅")


## Étape 1 — Pourquoi le texte est difficile pour une machine

Rappel du cours : un ordinateur ne comprend nativement que des **nombres**. Le langage humain, lui, est ambigu.

Exemple classique : *"Je vais à la banque"* → institution financière, ou bord d'une rivière ? Sans contexte, un mot peut avoir plusieurs sens.

Tout l'objectif du NLP est de transformer le texte en nombres exploitables, en essayant de conserver le plus d'information possible. C'est ce qu'on va faire, étape par étape, sur quelques phrases simples.


In [ ]:
# Nos phrases de travail pour tout le TP
phrases = [
    "Les chats mangent rapidement leurs croquettes !",
    "Le chien mange aussi des croquettes.",
    "Le chat dort toute la journée sur le canapé.",
]

for i, p in enumerate(phrases, start=1):
    print(f"Phrase {i} : {p}")


## Étape 2 — Le prétraitement du texte

On reprend les 4 étapes vues en cours (diapo 5), appliquées à notre première phrase :

1. **Tokenisation** — découper le texte en mots ("tokens").
2. **Minuscules** — uniformiser la casse ("Chat" = "chat").
3. **Mots vides (stopwords)** — retirer les mots peu informatifs : le, la, de, et...
4. **Lemmatisation / racinisation** — ramener un mot à sa forme de base ("mangent" → "manger").

On construit ces étapes une par une, comme dans l'exemple du cours.


In [ ]:
texte_brut = phrases[0]
print("1. Texte brut :")
print(texte_brut)


In [ ]:
# 2. Tokenisation : on découpe en mots (méthode simple avec une expression régulière)
tokens = re.findall(r"[a-zàâçéèêëîïôûùüÿñæœ]+", texte_brut.lower())
print("2. Tokenisation + minuscules :")
print(tokens)


In [ ]:
# 3. Suppression des stopwords (mots vides)
# Petite liste de stopwords français pour l'exercice (une liste plus complète existe dans NLTK)
stopwords_fr = {"le", "la", "les", "de", "des", "un", "une", "et", "leurs", "leur", "sur", "toute"}

tokens_sans_stopwords = [mot for mot in tokens if mot not in stopwords_fr]
print("3. Stopwords retirés :")
print(tokens_sans_stopwords)


In [ ]:
# 4. Lemmatisation simplifiée : on ramène quelques mots à leur forme de base
# (une vraie lemmatisation utiliserait spaCy ou NLTK ; ici on fait un dictionnaire simple pour comprendre le principe)
lemmes = {
    "chats": "chat",
    "mangent": "manger",
    "mange": "manger",
    "rapidement": "rapide",
    "croquettes": "croquette",
}

tokens_lemmatises = [lemmes.get(mot, mot) for mot in tokens_sans_stopwords]
print("4. Lemmatisation :")
print(tokens_lemmatises)


On retrouve exactement le résultat du cours : `[ chat, manger, rapide, croquette ]` 🎉

Maintenant qu'on a compris chaque étape séparément, on regroupe tout dans une seule fonction réutilisable.


In [ ]:
def nettoyer_texte(texte, retirer_stopwords=True, lemmatiser=True):
    """Nettoie une phrase : minuscules, tokenisation, (stopwords), (lemmatisation), retour en chaîne."""
    tokens = re.findall(r"[a-zàâçéèêëîïôûùüÿñæœ]+", texte.lower())
    if retirer_stopwords:
        tokens = [mot for mot in tokens if mot not in stopwords_fr]
    if lemmatiser:
        tokens = [lemmes.get(mot, mot) for mot in tokens]
    return " ".join(tokens)

phrases_propres = [nettoyer_texte(p) for p in phrases]

for original, propre in zip(phrases, phrases_propres):
    print(f"Avant : {original}")
    print(f"Après : {propre}\n")


## Étape 3 — Vectorisation : le Bag of Words

Rappel du cours (diapo 6) : la méthode la plus simple pour transformer du texte en nombres. On compte combien de fois chaque mot apparaît dans chaque phrase, sans se soucier de l'ordre.

On utilise `CountVectorizer` de scikit-learn, qui fait exactement ça automatiquement.


In [ ]:
count_vectorizer = CountVectorizer()
matrice_bow = count_vectorizer.fit_transform(phrases_propres)

# On affiche le résultat sous forme de tableau lisible, comme dans le cours
bow_df = pd.DataFrame(
    matrice_bow.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=[f"Phrase {i+1}" for i in range(len(phrases))],
)
bow_df


Chaque ligne est maintenant un **vecteur de nombres** : exactement ce dont un modèle de Machine Learning a besoin ! Mais on voit une limite : les mots très fréquents (comme "chat" ou "croquette" ici) pèsent autant que n'importe quel autre mot, même s'ils ne sont pas forcément les plus "caractéristiques".


## Étape 4 — Vectorisation : TF-IDF

Rappel du cours (diapo 7) : une version plus intelligente du Bag of Words.

- **TF** (Term Frequency) : à quel point un mot est fréquent *dans un document*.
- **IDF** (Inverse Document Frequency) : à quel point un mot est rare *dans l'ensemble des documents*.

Le TF-IDF combine les deux : un mot devient important s'il est fréquent dans un document, mais rare ailleurs.


In [ ]:
tfidf_vectorizer = TfidfVectorizer()
matrice_tfidf = tfidf_vectorizer.fit_transform(phrases_propres)

tfidf_df = pd.DataFrame(
    matrice_tfidf.toarray().round(3),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Phrase {i+1}" for i in range(len(phrases))],
)
tfidf_df


### Bag of Words vs TF-IDF : comparons

Regardez le mot **"chat"**, qui apparaît dans 2 des 3 phrases, contre le mot **"dort"**, qui n'apparaît que dans 1 phrase.

- En Bag of Words, les deux ont un score de `1` là où ils apparaissent : aucune distinction.
- En TF-IDF, le mot **"dort"** (rare, donc caractéristique) obtient un score plus élevé que **"chat"** (plus courant) dans la phrase où ils apparaissent tous les deux.

C'est exactement l'idée du cours : *"un mot devient important s'il est fréquent dans un document, mais rare ailleurs"*.


In [ ]:
comparaison = pd.DataFrame({
    "Bag of Words": bow_df["chat"],
    "TF-IDF": tfidf_df["chat"].round(3),
})
print("Mot : 'chat'")
comparaison


## Étape 5 — Les librairies Python pour le NLP

Rappel du cours (diapo 10) : quelques librairies incontournables. On vient déjà d'en utiliser une (scikit-learn) ; regardons maintenant chacune un peu plus en détail, avec un petit exemple concret quand c'est possible.

| Librairie | Ce qu'elle apporte | Niveau |
|---|---|---|
| **scikit-learn** | `CountVectorizer`, `TfidfVectorizer`, `NearestNeighbors`... déjà utilisée dans ce TP | Débutant |
| **NLTK** | Une des plus anciennes librairies NLP : tokenisation, stopwords, stemming | Débutant |
| **spaCy** | Rapide et moderne, idéale pour la vraie lemmatisation et l'analyse grammaticale | Intermédiaire |
| **Hugging Face** | Modèles de langage pré-entraînés (BERT, GPT...) pour aller plus loin | Avancé |


### 5.1 — scikit-learn (déjà utilisée)

C'est celle qu'on a utilisée depuis le début de ce TP, avec `CountVectorizer` (Bag of Words) et `TfidfVectorizer` (TF-IDF). Elle contient aussi `NearestNeighbors`, qui permet de retrouver rapidement les éléments les plus proches d'un vecteur — utile pour comparer des textes entre eux (on la retrouvera dans un futur projet).


### 5.2 — NLTK

**NLTK** (Natural Language Toolkit) est l'une des toutes premières librairies de NLP en Python. Elle fournit, entre autres, une tokenisation plus robuste que notre petite expression régulière maison, et des listes de stopwords toutes prêtes pour de nombreuses langues (dont le français).

On compare ci-dessous notre tokenisation "maison" (Étape 2) avec celle de NLTK, et notre petite liste de stopwords avec la vraie liste NLTK.


In [ ]:
# NLTK est en général déjà installé sur Colab ; sinon : !pip install nltk
import nltk
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords as nltk_stopwords

phrase_test = phrases[0]

print("Notre tokenisation maison (regex) :")
print(re.findall(r"[a-zàâçéèêëîïôûùüÿñæœ]+", phrase_test.lower()))

print("\nTokenisation avec NLTK (word_tokenize) :")
print(word_tokenize(phrase_test, language="french"))


In [ ]:
stopwords_nltk_fr = set(nltk_stopwords.words("french"))

print(f"Notre liste maison de stopwords : {len(stopwords_fr)} mots")
print(f"Liste de stopwords NLTK (français) : {len(stopwords_nltk_fr)} mots")

print("\nQuelques mots présents dans NLTK mais pas dans notre liste maison :")
print(sorted(stopwords_nltk_fr - stopwords_fr)[:10])


On voit tout de suite l'intérêt d'une vraie librairie : notre liste maison ne contenait que quelques mots choisis à la main pour l'exemple, alors que NLTK en fournit plus de 150, déjà validés, pour le français.


### 5.3 — spaCy

**spaCy** est une librairie plus récente et plus rapide que NLTK, particulièrement bonne pour la **vraie lemmatisation** (contrairement à notre dictionnaire fait main à l'Étape 2, qui ne connaissait que 5 mots !) et pour l'analyse grammaticale (identifier si un mot est un nom, un verbe, etc.).

Elle nécessite de télécharger un petit modèle pré-entraîné pour le français (~15 Mo, une seule fois).


In [ ]:
# Installation (une seule fois, peut prendre 1-2 minutes) :
# !pip install -U spacy
# !python -m spacy download fr_core_news_sm

import spacy

nlp_spacy = spacy.load("fr_core_news_sm")
doc = nlp_spacy(phrase_test)

print(f"{'Mot':<15}{'Lemme (spaCy)':<18}{'Nature grammaticale'}")
for token in doc:
    print(f"{token.text:<15}{token.lemma_:<18}{token.pos_}")


Contrairement à notre dictionnaire manuel de l'Étape 2 (qui ne fonctionnait que pour 4-5 mots prévus à l'avance), spaCy sait lemmatiser **n'importe quel mot** du français, et donne en plus sa nature grammaticale (nom, verbe, adjectif...). C'est ce type d'outil qu'on utiliserait pour une vraie lemmatisation en production.


### 5.4 — Hugging Face

**Hugging Face** donne accès à des milliers de modèles de langage pré-entraînés (BERT, GPT, CamemBERT pour le français...) via sa librairie `transformers`. Plutôt que de compter des mots (Bag of Words, TF-IDF), ces modèles ont une vraie compréhension du contexte.

C'est le niveau "avancé" du cours (voir *Pour aller plus loin*) — on se contente ici d'un simple aperçu avec `pipeline()`, l'API la plus simple de la librairie, qui charge un modèle déjà entraîné pour une tâche donnée (ici, l'analyse de sentiment).

> ⚠️ **Optionnel** — cette cellule télécharge un modèle (quelques centaines de Mo) lors de la première exécution, ce qui peut prendre 1 à 2 minutes. Vous pouvez la sauter sans impact sur la suite du TP.


In [ ]:
# ⚠️ Optionnel — nécessite : !pip install transformers torch --quiet
# from transformers import pipeline
#
# classifieur_sentiment = pipeline("sentiment-analysis")
# resultats = classifieur_sentiment([
#     "I love this book, it was amazing!",
#     "This was a terrible experience.",
# ])
# for r in resultats:
#     print(r)

# Note : le modèle par défaut est entraîné en anglais. Pour du français, on utiliserait
# par exemple un modèle comme "tblard/tf-allocine", disponible sur huggingface.co/models.


## 🎯 Exercices — À vous de jouer !

C'est le **TP à proprement parler** : vous allez répéter tout le pipeline sur vos propres phrases, pour bien vous l'approprier.

### Exercice 1 — Vos propres phrases

Remplacez le contenu de `mes_phrases` ci-dessous par **3 phrases de votre choix** (en français, avec au moins un mot qui revient dans plusieurs phrases), puis appliquez `nettoyer_texte()` à chacune.


In [ ]:
# TODO Exercice 1
# mes_phrases = [
#     "...",
#     "...",
#     "...",
# ]
# mes_phrases_propres = [nettoyer_texte(p) for p in mes_phrases]
# for original, propre in zip(mes_phrases, mes_phrases_propres):
#     print(f"Avant : {original}")
#     print(f"Après : {propre}\n")


### Exercice 2 — Bag of Words sur vos phrases

Appliquez un `CountVectorizer` sur `mes_phrases_propres` et affichez le tableau résultat (comme à l'Étape 3). Combien de mots uniques compte votre vocabulaire ?


In [ ]:
# TODO Exercice 2
# mon_count_vectorizer = CountVectorizer()
# ma_matrice_bow = mon_count_vectorizer.fit_transform(mes_phrases_propres)
# pd.DataFrame(
#     ma_matrice_bow.toarray(),
#     columns=mon_count_vectorizer.get_feature_names_out(),
#     index=[f"Phrase {i+1}" for i in range(len(mes_phrases))],
# )


### Exercice 3 — TF-IDF sur vos phrases

Faites la même chose avec `TfidfVectorizer`. Repérez un mot qui apparaît dans **une seule** de vos phrases et un mot qui apparaît dans **toutes**. Comparez leurs scores TF-IDF : le premier doit être plus élevé que le second.


In [ ]:
# TODO Exercice 3
# mon_tfidf_vectorizer = TfidfVectorizer()
# ma_matrice_tfidf = mon_tfidf_vectorizer.fit_transform(mes_phrases_propres)
# pd.DataFrame(
#     ma_matrice_tfidf.toarray().round(3),
#     columns=mon_tfidf_vectorizer.get_feature_names_out(),
#     index=[f"Phrase {i+1}" for i in range(len(mes_phrases))],
# )


### Exercice 4 — Avec ou sans stopwords ?

Relancez le nettoyage de vos phrases avec `nettoyer_texte(phrase, retirer_stopwords=False)` (donc **sans** retirer les stopwords), puis reconstruisez le Bag of Words. Que se passe-t-il sur la taille du vocabulaire et sur les scores ? Les mots comme "le", "de", "et" prennent-ils trop de place ?


In [ ]:
# TODO Exercice 4
# mes_phrases_avec_stopwords = [nettoyer_texte(p, retirer_stopwords=False) for p in mes_phrases]
# vectorizer_test = CountVectorizer()
# matrice_test = vectorizer_test.fit_transform(mes_phrases_avec_stopwords)
# print("Taille du vocabulaire AVEC stopwords :", len(vectorizer_test.get_feature_names_out()))
# print("Taille du vocabulaire SANS stopwords  :", len(count_vectorizer.get_feature_names_out()) if False else "à comparer avec l'exercice 2")


### Exercice 5 (bonus) — Ajoutez une 4e phrase

Ajoutez une 4e phrase à votre liste qui réutilise **un mot déjà présent** dans une de vos 3 premières phrases. Recalculez le TF-IDF sur les 4 phrases. Le score de ce mot a-t-il baissé par rapport à avant (quand il n'apparaissait que dans 1 seule phrase) ? Pourquoi, d'après la définition de l'IDF ?


In [ ]:
# TODO Exercice 5 (bonus)
# mes_phrases_v2 = mes_phrases + ["..."]
# mes_phrases_v2_propres = [nettoyer_texte(p) for p in mes_phrases_v2]
# vectorizer_v2 = TfidfVectorizer()
# matrice_v2 = vectorizer_v2.fit_transform(mes_phrases_v2_propres)
# pd.DataFrame(
#     matrice_v2.toarray().round(3),
#     columns=vectorizer_v2.get_feature_names_out(),
#     index=[f"Phrase {i+1}" for i in range(len(mes_phrases_v2))],
# )


### Exercice 6 (bonus) — Comparer avec de vraies librairies

Reprenez vos phrases de l'Exercice 1 et :

1. Tokenisez-les avec `word_tokenize()` de NLTK au lieu de la regex maison. Le résultat est-il différent ?
2. Lemmatisez-les avec `nlp_spacy(...)` au lieu du petit dictionnaire manuel de l'Étape 2. Est-ce que ça fonctionne même sur des mots qu'on n'avait pas prévus ?


In [ ]:
# TODO Exercice 6 (bonus)
# for phrase in mes_phrases:
#     print(phrase, "->", word_tokenize(phrase, language="french"))

# for phrase in mes_phrases:
#     doc = nlp_spacy(phrase)
#     print(phrase, "->", [token.lemma_ for token in doc])


## Conclusion

Vous venez de manipuler de vos propres mains les deux premières briques du pipeline NLP :

**Texte brut → Prétraitement → Vectorisation (Bag of Words / TF-IDF)**

Vous savez maintenant :

- pourquoi on nettoie le texte avant de le vectoriser,
- comment fonctionne le comptage de mots (Bag of Words),
- pourquoi le TF-IDF est plus pertinent pour mettre en valeur les mots caractéristiques.

### Et après ?

Les étapes suivantes du pipeline — **modèle** et **résultat** (diapo 4), avec par exemple la similarité cosinus pour comparer des textes entre eux — seront vues dans un prochain TP/projet.

### Pour aller plus loin

- **NLTK** et **spaCy** : des librairies plus complètes pour la tokenisation, les stopwords et la vraie lemmatisation.
- **Word embeddings** (Word2Vec, GloVe) : représenter le sens des mots, pas seulement les compter.
- **Transformers** (BERT, GPT) et **Hugging Face** : les modèles de langage modernes.

Bon TP ! 🚀
